In [ ]:
!pip install requests beautifulsoup4 lxml pandas tqdm

In [ ]:
!pip install playwright
!playwright install

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 20.7 MB/s eta 0:00:00
(node:1577) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 344.7s167.3 MiB [] 0% 101.9s167.3 MiB [] 0% 182.3s167.3 MiB [] 0% 532.4s167.3 MiB [] 0% 447.4s167.3 MiB [] 0% 597.8s167.3 MiB [] 0% 526.6s167.3 MiB [] 0% 478.8s167.3 MiB [] 0% 444.3s167.3 MiB [] 0% 416.3s167.3 MiB [] 0% 393.9s167.3 MiB [] 0% 376.5s167.3 MiB [] 0% 361.1s167.3 MiB [] 0% 348.1s167.3 MiB [] 0% 336.2s167.3 MiB [] 0% 312.8s167.3 MiB [] 0% 289.7s167.3 MiB [] 0% 283.6s167.3 MiB [] 0% 254.6s167.3 MiB [] 0% 232.1s167.3 MiB [] 0% 214.4s167.3 MiB [] 0% 200.1s167.3 MiB [] 0% 188.7s167.3 MiB [] 0% 179.7s167.3 MiB [] 0% 166.8s167.3 MiB [] 0% 155.9s167.3 MiB [] 0% 144.1s167.3 MiB [] 0% 134.0s167.3

In [ ]:
!playwright install-deps

Installing dependencies...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,797 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,772 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/u

In [ ]:
import asyncio
from playwright.async_api import async_playwright # Changed to async_playwright
from bs4 import BeautifulSoup
import json

USERNAME = "Ryussi"
PASSWORD = "Issuyr"

LOGIN_URL = "https://docs.mosmb.com/index.php?title=Special:UserLogin"
START_URL = "https://docs.mosmb.com/index.php/MoSMB_Introduction"

visited = set()
to_visit = [START_URL]
all_urls = set() # Initialize all_urls here

def is_valid(url):
    return (
        "/index.php/" in url and
        "Special:" not in url and
        ":" not in url.split("/index.php/")[-1]  # remove namespaces
    )

async def scrape_site_with_login(): # Wrapped scraping logic in an async function
    async with async_playwright() as p: # Changed to async_playwright
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # LOGIN
        await page.goto(LOGIN_URL)
        try:
            await page.wait_for_selector('#wpName1', timeout=20000) # Wait for username input
        except Exception as e:
            print(f"Timeout waiting for username input (#wpName1) on login page: {e}")
            await browser.close()
            return # Exit if login form not found

        await page.fill('#wpName1', USERNAME)
        await page.fill('#wpPassword1', PASSWORD)
        await page.click('#wpLoginAttempt')
        await page.wait_for_load_state('networkidle')

        print("✅ Logged in")

        while to_visit:
            url = to_visit.pop(0)

            if url in visited:
                continue

            print(f"Scraping: {url}")
            visited.add(url)
            all_urls.add(url) # Add to all_urls as well

            try:
                await page.goto(url)
                await asyncio.sleep(2) # Use asyncio.sleep

                html = await page.content()
                soup = BeautifulSoup(html, "lxml")

                for a in soup.find_all("a", href=True):
                    link = a["href"]

                    if link.startswith("/index.php/"):
                        full_url = "https://docs.mosmb.com" + link

                        if is_valid(full_url) and full_url not in visited and full_url not in to_visit:
                            to_visit.append(full_url)
                            all_urls.add(full_url)

            except Exception as e:
                print("Error:", e)

        await browser.close()

await scrape_site_with_login() # Call the async function

# SAVE URLs
print("\nTotal URLs:", len(all_urls))

with open("mosmb_all_urls.json", "w") as f:
    json.dump(list(all_urls), f, indent=2)

print("Saved URLs")

✅ Logged in
Scraping: https://docs.mosmb.com/index.php/MoSMB_Introduction
Scraping: https://docs.mosmb.com/index.php/MoSMB_Installation
Scraping: https://docs.mosmb.com/index.php/MoSMB_Documentation
Scraping: https://docs.mosmb.com/index.php/Getting_Started_with_MoSMB
Scraping: https://docs.mosmb.com/index.php/Release_Notes
Scraping: https://docs.mosmb.com/index.php/Server_Mode_-_Standalone
Scraping: https://docs.mosmb.com/index.php/Server_Mode_-_Cluster_-_Single_Domain_Clustering
Scraping: https://docs.mosmb.com/index.php/Server_Mode_-_Cluster_-_Multi_Domain_Clustering
Scraping: https://docs.mosmb.com/index.php/Kerberos_(krb5)_Authentication_Mode
Scraping: https://docs.mosmb.com/index.php/NTLMSSP_Authentication_Support
Scraping: https://docs.mosmb.com/index.php/Ntlm_(ntlmLocal)_Authentication_Mode
Scraping: https://docs.mosmb.com/index.php/Uid_gid_Mapping
Scraping: https://docs.mosmb.com/index.php/Authorization_Mode_-_none
Scraping: https://docs.mosmb.com/index.php/Authorization_Mode_

In [ ]:
import json
import re

# =========================
# LOAD DATA
# =========================
with open("/content/mosmb_docs_content.json") as f:
    docs = json.load(f)

# =========================
# CLEAN TEXT
# =========================
def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ")
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    return text.strip()

# =========================
# NOISE FILTER
# =========================
def is_noise(text):
    if not text:
        return True
    if len(text) < 30:
        return True
    if len(text.split()) < 5:
        return True
    if text.lower() in ["contents", "example", "note"]:
        return True
    return False

# =========================
# LOW VALUE FILTER
# =========================
def is_low_value(text):
    return (
        len(text.split()) < 6
        or (text.count("=") > 4 and len(text.split()) < 12)
    )

# =========================
# CODE DETECTION (IMPROVED)
# =========================
def is_code(text):
    patterns = [
        r'=', r'/etc/', r'\{', r'\}', r'\[', r'\]',
        r'command', r'output', r'example',
        r'\bchmod\b', r'\bcat\b', r'\bservice\b',
        r'\bmosmb\b', r'\brping\b', r'\bsssd\b',
        r'\.conf', r'\.json', r'\.log'
    ]
    score = sum(bool(re.search(p, text.lower())) for p in patterns)
    return score >= 2

# =========================
# SPLIT MIXED CONTENT
# =========================
def split_mixed_content(text):
    splits = re.split(r'(Command:|Output:|Example:)', text)

    parts = []
    current = ""

    for part in splits:
        if part in ["Command:", "Output:", "Example:"]:
            if current.strip():
                parts.append(current.strip())
            current = part
        else:
            current += " " + part

    if current.strip():
        parts.append(current.strip())

    return parts

# =========================
# SMART SPLIT
# =========================
def smart_split(text, min_chars=120, max_chars=600):
    sentences = re.split(r'(?<=[.!?]) +', text)

    chunks = []
    current = ""

    for s in sentences:
        if len(current) + len(s) <= max_chars:
            current += " " + s
        else:
            if len(current.strip()) >= min_chars:
                chunks.append(current.strip())
            current = s

    if len(current.strip()) >= min_chars:
        chunks.append(current.strip())

    return chunks

# =========================
# MAIN PIPELINE
# =========================
final_chunks = []

for doc in docs:
    url = doc.get("url", "")
    raw_text = doc.get("text", "")

    text = clean_text(raw_text)

    if is_noise(text):
        continue

    # STEP 1: Split mixed content
    raw_parts = split_mixed_content(text)

    for block in raw_parts:
        block = block.strip()

        if is_noise(block):
            continue

        # =========================
        # CODE HANDLING
        # =========================
        if is_code(block):
            if not is_low_value(block):
                final_chunks.append({
                    "url": url,
                    "type": "code",
                    "text": block,
                    "source": url
                })
            continue

        # =========================
        # TEXT HANDLING
        # =========================
        parts = smart_split(block)

        for p in parts:
            p = p.strip()

            if not p:
                continue

            if is_low_value(p):
                continue

            final_chunks.append({
                "url": url,
                "type": "text",
                "text": p,
                "source": url
            })

# =========================
# SAVE OUTPUT
# =========================
print("✅ Total chunks:", len(final_chunks))

with open("mosmb_final_chunks.json", "w") as f:
    json.dump(final_chunks, f, indent=2)

print("✅ Saved as mosmb_final_chunks.json")

✅ Total chunks: 1107
✅ Saved as mosmb_final_chunks.json


In [ ]:
import json
import random

with open("mosmb_final_chunks.json") as f:
    chunks = json.load(f)

for chunk in random.sample(chunks, 5):
    print("\n--------------------")
    print(chunk)


--------------------
{'url': 'https://docs.mosmb.com/index.php/Troubleshoot_Guide', 'type': 'text', 'text': "Manage File Shares with Shared Folders MMC Snap-in. Close sessions and close open file handles can be managed via MMC Open mmc on a windows   Navigate to the Start Menu and select Run, type 'mmc'. On the MMC console from File menu select Add or Remove snap-ins In Shared Folder properties   Enter the node fqdn or ip and connect. Check Sessions to list the smb-sessions connected to the node. Check Open Files to list all the open files on the node.", 'source': 'https://docs.mosmb.com/index.php/Troubleshoot_Guide'}

--------------------
{'url': 'https://docs.mosmb.com/index.php/Network_Configuration', 'type': 'text', 'text': 'Under Forward Lookup Zones , select the desired domain name. For example .dom. Right click in the Window. Select New Host (A or AAAA)... Fill in the related information: Name is the hostname of the Mo SMB server. IP address is the IP address of the Mo SMB serv

In [ ]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import json
import re

# =========================
# CONFIG
# =========================
USERNAME = "Ryussi"
PASSWORD = "Issuyr"

LOGIN_URL = "https://docs.mosmb.com/index.php?title=Special:UserLogin"
TARGET_URL = "https://docs.mosmb.com/index.php/MoSMB_Error_Codes"

# =========================
# HELPERS
# =========================
def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_all_error_codes(text):
    return re.findall(r'\b[0-9A-Fa-f]{8}\b', text)

def split_solution(text):
    # DO NOT break URLs
    parts = re.split(r'\n|\.\s+(?!org|com|html)', text)
    return [clean_text(p) for p in parts if len(p.strip()) > 10]

# =========================
# PARSER (FIXED PROPERLY)
# =========================
def parse_error_tables(soup):
    results = []

    current_category = None

    content = soup.find("div", {"id": "mw-content-text"})
    if not content:
        return results

    for element in content.find_all(["h2", "h3", "table"]):

        # CATEGORY
        if element.name in ["h2", "h3"]:
            heading = clean_text(element.get_text())

            if heading.lower() in ["contents", "how to read the table"]:
                continue

            category_match = re.match(r'^[A-Z]+', heading.upper())
            if category_match:
                current_category = category_match.group(0)

            continue

        # TABLE PARSING
        if element.name == "table":
            rows = element.find_all("tr")

            active_error_string = None
            active_solution = None

            rowspan_error = 0
            rowspan_solution = 0

            for row in rows:
                cols = row.find_all("td")

                if not cols:
                    continue

                # =========================
                # ERROR CODES
                # =========================
                raw_code_text = clean_text(cols[0].get_text())
                error_codes = extract_all_error_codes(raw_code_text)

                if not error_codes:
                    continue

                # =========================
                # ERROR STRING HANDLING
                # =========================
                if len(cols) >= 2:
                    active_error_string = clean_text(cols[1].get_text())

                    if cols[1].has_attr("rowspan"):
                        rowspan_error = int(cols[1]["rowspan"])
                    else:
                        rowspan_error = 1

                elif rowspan_error > 0:
                    rowspan_error -= 1

                # =========================
                # SOLUTION HANDLING
                # =========================
                if len(cols) >= 3:
                    active_solution = clean_text(cols[2].get_text())

                    if cols[2].has_attr("rowspan"):
                        rowspan_solution = int(cols[2]["rowspan"])
                    else:
                        rowspan_solution = 1

                elif rowspan_solution > 0:
                    rowspan_solution -= 1

                # =========================
                # FINAL ENTRY
                # =========================
                results.append({
                    "error_codes": error_codes,
                    "error_string": active_error_string,
                    "category": current_category,
                    "solution": split_solution(active_solution) if active_solution else [],
                    "source": TARGET_URL
                })

    return results

# =========================
# MAIN
# =========================
async def scrape_error_codes():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # LOGIN
        await page.goto(LOGIN_URL)
        await page.fill('input[name="wpName"]', USERNAME)
        await page.fill('input[name="wpPassword"]', PASSWORD)
        await page.click('button[name="wploginattempt"]')
        await page.wait_for_load_state('networkidle')

        print("✅ Logged in")

        # TARGET PAGE
        await page.goto(TARGET_URL)
        await asyncio.sleep(3)

        html = await page.content()
        soup = BeautifulSoup(html, "lxml")

        data = parse_error_tables(soup)

        await browser.close()

        return data

# =========================
# RUN
# =========================
data = await scrape_error_codes()

print("Before grouping:", len(data))

# =========================
# GROUPING (FINAL FIX)
# =========================
grouped = {}

for item in data:
    key = (
        item["error_string"],
        item["category"],
    )

    if key not in grouped:
        grouped[key] = {
            "error_codes": set(),
            "error_string": item["error_string"],
            "category": item["category"],
            "solution": set(),
            "source": item["source"]
        }

    grouped[key]["error_codes"].update(item["error_codes"])
    grouped[key]["solution"].update(item["solution"])

# Convert sets → lists
final_data = []
for v in grouped.values():
    final_data.append({
        "error_codes": list(v["error_codes"]),
        "error_string": v["error_string"],
        "category": v["category"],
        "solution": list(v["solution"]),
        "source": v["source"]
    })

print("After grouping:", len(final_data))

# =========================
# SAVE CORRECT OUTPUT
# =========================
with open("mosmb_error_structured.json", "w") as f:
    json.dump(final_data, f, indent=2)

print("✅ Saved as mosmb_error_structured.json")

✅ Logged in
Before grouping: 701
After grouping: 652
✅ Saved as mosmb_error_structured.json


In [ ]:
!pip install sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()

collection = client.create_collection(
    name="mosmb_docs_upt_01"
)

AttributeError: module 'sympy' has no attribute 'core'

In [ ]:
import json

with open("/content/drive/MyDrive/mosmb_rag/data/mosmb_final_chunks.json") as f:
    docs = json.load(f)

In [ ]:
documents = []
metadatas = []
ids = []

for i, d in enumerate(docs):

    documents.append(d["text"])

    metadatas.append({
        "source": d["url"],
        "type": d["type"]
    })

    ids.append(str(i))

In [ ]:
embeddings = model.encode(documents).tolist()

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

In [ ]:
import re

def find_error_code(query):

    match = re.search(r'\b[0-9A-Fa-f]{8}\b', str(query))

    if match:
        return match.group().upper()

    return None

In [ ]:
import json

with open("/content/drive/MyDrive/mosmb_rag/data/mosmb_error_structured.json") as f:
    error_db = json.load(f)

In [ ]:
# =========================
# LOAD ERROR DATA
# =========================
with open("/content/drive/MyDrive/mosmb_rag/data/mosmb_error_structured.json") as f:
    error_db = json.load(f)

error_documents = []
error_metadatas = []
error_ids = []

for i, item in enumerate(error_db):

    text = f"""
Error Code: {", ".join(item["error_codes"])}
Description: {item["error_string"]}
Solution:
{chr(10).join(item["solution"])}
"""

    error_documents.append(text)

    error_metadatas.append({
        "type": "error",                      # IMPORTANT
        "error_codes": item["error_codes"],   # LIST
        "category": item["category"],
        "source": item["source"]
    })

    error_ids.append(f"error_{i}")

In [ ]:
# Add docs (already exists)
if collection.count() == 0:

    doc_embeddings = model.encode(documents).tolist()

    collection.add(
        documents=documents,
        embeddings=doc_embeddings,
        metadatas=metadatas,
        ids=ids
    )

    print("Documents added")

    # =========================
    # ADD ERROR DATA ALSO
    # =========================

    error_embeddings = model.encode(error_documents).tolist()

    collection.add(
        documents=error_documents,
        embeddings=error_embeddings,
        metadatas=error_metadatas,
        ids=error_ids
    )

    print("Error data added")

else:
    print("Chroma already populated")

Documents added
Error data added


In [ ]:
def get_error_from_chroma(code):

    results = collection.get(
        where={"type": "error"}
    )

    docs = results["documents"]
    metas = results["metadatas"]

    for doc, meta in zip(docs, metas):
        if code in meta["error_codes"]:
            return doc, meta

    return None

In [ ]:
error_lookup = {}

for item in error_db:
    for code in item["error_codes"]:
        error_lookup[code] = item

In [ ]:
def vector_search(query):

    query_embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=20
    )

    return results["documents"][0]

In [ ]:
def rerank(query, documents):

    pairs = [[query, doc] for doc in documents]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, documents),
        reverse=True
    )

    best_docs = [doc for score, doc in ranked[:5]]

    return best_docs

In [ ]:
def retrieve(query):

    # Step 1 — detect error code
    code = find_error_code(query)

    if code:

        result = get_error_from_chroma(code)

        if result:
            doc, meta = result

            print("\n----------------")
            print("Error Code:", code)
            print(doc)

            return

    # Step 2 — vector search
    docs = vector_search(query)

    # Step 3 — rerank
    best_docs = rerank(query, docs)

    print("\nTop Results:\n")

    for doc in best_docs:

        print("\n----------------")
        print(doc)

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

embedding_model = model
retrieve("How to configure kerberos for mosmb?")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Top Results:


----------------
Configure the Kerberos settings by obtaining the Kerberos ticket from the Ticket Granting server (AD-KDC Server). Ensure that you have mentioned the correct storage path in /etc/mosmb/mosmb.json in Share Configuration . Start / Restart the Mo SMB server for the desired changes to take effect.

----------------
All Mo SMB-SA builds are set to use the "Standalone Mode" of operation by default. To use Mo SMB server in Standalone Mode with Kerberos authentication simply follow the configurations mentioned in steps below one by one. 1. Configure Network Time Protocol (NTP) Configuration settings on Mo SMB server node. 2. Install Kerberos packages on mosmb server 3. Request the Domain Administration team for AD and DNS Configurations. Configure name resolution settings on Mo SMB server.

----------------
1. Mention one of the Domain in as default in krb5.conf file out of all domains to be configured. 2. MoSMB will support only cache type FILE for ' kerberos v

In [ ]:
retrieve("01010000")


NameError: name 'retrieve' is not defined

In [ ]:
query = "How to add NTLMSSP support"

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=5
)

In [ ]:
for doc in results["documents"][0]:

    print("\n----------------")
    print(doc)


----------------
1 Add NTLMSSP support 1.1 Machine Password 1.2 mosmb_sssd.conf / sssd.conf whichever is used, make changes 1.3 Service Restarts 1.2 mosmb_sssd.conf / sssd.conf whichever is used, make changes

----------------
For release versions:3.1.1_000 refer - Mo SMB Kerberos Ticket Generation For release versions:3.1.1_001 and above refer - Mo SMB Kerberos Ticket Generation. 5. Mo SMB server now supports NTLMSSP authentication from Release versions:3.1.1_004 and above. For each Mo SMB server node in cluster, carry out these changes to get the NTLMSSP authentication support working. 6. Mo SMB server is Zoo Keeper client; Perform Mo SMB -Zoo Keeper settings 8.

----------------
For release versions:3.1.1_000 refer - Mo SMB Kerberos Ticket Generation For release versions:3.1.1_001 and above refer - Mo SMB Kerberos Ticket Generation. 5. Mo SMB server now supports NTLMSSP authentication from Release versions:3.1.1_004 and above. For each Mo SMB server node in cluster, carry out these

In [ ]:
import os

os.makedirs("/content/drive/MyDrive/mosmb_rag/data/chroma_db", exist_ok=True)

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
import json

client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/mosmb_rag/data/chroma_db"
)

collection = client.get_or_create_collection(
    name="mosmb_docs_upt_01"
)

model = SentenceTransformer("all-MiniLM-L6-v2")

with open("/content/drive/MyDrive/mosmb_rag/data/mosmb_final_chunks.json") as f:
    docs = json.load(f)

documents = []
metadatas = []
ids = []

for i, d in enumerate(docs):

    documents.append(d["text"])

    metadatas.append({
        "source": d["url"],
        "type": d["type"]
    })

    ids.append(str(i))


# Only add if DB is empty
if collection.count() == 0:

    embeddings = model.encode(documents).tolist()

    collection.add(
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )

    print("Documents added to Chroma DB")

else:

    print("Chroma DB already populated.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Documents added to Chroma DB


In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
import json

def insert_error_data_to_chroma():

    # =========================
    # CONNECT TO EXISTING DB
    # =========================
    client = chromadb.PersistentClient(
        path="/content/drive/MyDrive/mosmb_rag/data/chroma_db"
    )

    collection = client.get_or_create_collection(
        name="mosmb_docs_upt_01"
    )

    model = SentenceTransformer("all-MiniLM-L6-v2")

    # =========================
    # CHECK IF ERROR DATA ALREADY EXISTS
    # =========================
    existing_errors = collection.get(where={"type": "error"})

    if len(existing_errors["ids"]) > 0:
        print("✅ Error data already exists in Chroma DB")
        print("Count:", len(existing_errors["ids"]))
        return

    print("🚀 Inserting error data into Chroma DB...")

    # =========================
    # LOAD ERROR JSON
    # =========================
    with open("/content/drive/MyDrive/mosmb_rag/data/mosmb_error_structured.json") as f:
        error_db = json.load(f)

    error_documents = []
    error_metadatas = []
    error_ids = []

    # =========================
    # PREPARE DATA
    # =========================
    for i, item in enumerate(error_db):

        text = f"""
Error Code: {", ".join(item["error_codes"])}
Description: {item["error_string"]}
Solution:
{chr(10).join(item["solution"])}
"""

        error_documents.append(text)

        error_metadatas.append({
            "type": "error",
            "error_codes": [code.strip().upper() for code in item["error_codes"]],
            "category": item["category"],
            "source": item["source"]
        })

        error_ids.append(f"error_{i}")

    # =========================
    # CREATE EMBEDDINGS
    # =========================
    embeddings = model.encode(error_documents).tolist()

    # =========================
    # INSERT INTO CHROMA
    # =========================
    collection.add(
        documents=error_documents,
        embeddings=embeddings,
        metadatas=error_metadatas,
        ids=error_ids
    )

    print("✅ Error data successfully added to Chroma DB")
    print("Inserted:", len(error_ids))


# =========================
# RUN FUNCTION
# =========================
insert_error_data_to_chroma()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 Inserting error data into Chroma DB...
✅ Error data successfully added to Chroma DB
Inserted: 701


In [ ]:
print(collection.count())

1808


In [ ]:
import json

with open("/content/drive/MyDrive/mosmb_rag/data/mosmb_final_chunks.json") as f:
    docs = json.load(f)

print("Total chunks in JSON:", len(docs))

Total chunks in JSON: 1107


In [ ]:
print("Error count:",
      len(collection.get(where={"type": "error"})["ids"]))

Error count: 701


In [ ]:
retrieve("I am getting 02000100 error")


----------------
Error Code: 02000100

Error Code: 02000100
Description: Unexpected lstat failed errno %d, Please check if you can access this path %s , also check if File System is accessible
Solution:
Please check if File System is accessible



In [ ]:
# Install the tree utility
!apt-get install tree

# Print the structure
!tree /content/drive/MyDrive/mosmb_rag

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 5 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (104 kB/s)
Selecting previously unselected package tree.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...
/content/drive/MyDrive/mosmb_rag
├── data
│   ├── chroma_db
│   │   ├── chroma.sqlite3
│   │   └── e4298d9d-a563-41bc-85d7-040e3b573a97
│   │       ├── data_level0.bin
│   │       ├── header.bin
│   │       ├── index_metadata.pickle
│   │       ├── length.bin
│   │       └── link_lists.bi

In [ ]:
print(collection.count())
print(len(error_db))

1808
701


In [ ]:
!nvidia-smi
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

Sun Mar 15 19:39:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvidia-smi
torch.cuda.is_available()
next(llm.parameters()).device

Sun Mar 15 18:21:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

NameError: name 'llm' is not defined

In [ ]:
!pip install transformers accelerate bitsandbytes
!pip install sentence-transformers
!pip install rank-bm25
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Fo

In [ ]:
!pip install huggingface_hub

from huggingface_hub import login

login()

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

llm = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config,
    torch_dtype=torch.float16 # Keep this if previous deprecation warning referred to 'dtype' and 'torch_dtype' interchangeably and `torch_dtype` is actually expected here, or remove it if `dtype` is universally preferred.
)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [ ]:
import chromadb

# Initialize a persistent ChromaDB client, specifying the path to your database files
client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/mosmb_rag/data/chroma_db" # This path should match where you want to store your ChromaDB data
)

# Get or create a collection within the ChromaDB instance
# If the collection 'mosmb_docs_upt_01' already exists, it will be retrieved.
# If it doesn't exist, a new one will be created.
collection = client.get_or_create_collection(
    name="mosmb_docs_upt_01"
)

print(f"ChromaDB collection '{collection.name}' is ready with {collection.count()} documents.")


ChromaDB collection 'mosmb_docs_upt_01' is ready with 1808 documents.


In [ ]:
from rank_bm25 import BM25Okapi
import json

with open("/content/drive/MyDrive/mosmb_rag/data/mosmb_final_chunks.json") as f:
    docs = json.load(f)

corpus = [d["text"] for d in docs]

tokenized_corpus = [doc.split() for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def vector_search(query):

    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=10,
        where={"type": "doc"},
        include=['documents']
    )

    return results["documents"][0]  # ✅ list of strings

In [ ]:
def bm25_search(query):

    tokenized_query = query.split()

    scores = bm25.get_scores(tokenized_query)

    top_n_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:10]

    return [docs[i]['text'] for i in top_n_indices]  # ✅ strings only

In [ ]:
def clean_output(text):

    # Remove garbage phrases
    garbage_phrases = [
        "Best regards",
        "I hope this helps",
        "Please let me know",
        "I will be happy to help",
        "Thank you"
    ]

    for g in garbage_phrases:
        text = text.replace(g, "")

    # Remove repeated lines
    lines = text.split("\n")
    seen = set()
    cleaned_lines = []

    for line in lines:
        line = line.strip()
        if line and line not in seen:
            cleaned_lines.append(line)
            seen.add(line)

    return "\n".join(cleaned_lines)

In [ ]:
def clean_context(docs):

    seen = set()
    cleaned = []

    for doc in docs:

        doc = doc.strip()

        # 🔥 remove junk
        if len(doc) < 50:
            continue

        key = doc.lower()

        if key not in seen:
            seen.add(key)
            cleaned.append(doc)

    return cleaned[:5]  # 🔥 reduce context size

In [ ]:
def hybrid_retrieval(query):

    vec_docs = vector_search(query)
    bm_docs = bm25_search(query)

    # 🔥 normalize everything to string
    def normalize(doc):
        if isinstance(doc, dict):
            return doc.get("text", "")
        return doc

    vec_docs = [normalize(d) for d in vec_docs]
    bm_docs = [normalize(d) for d in bm_docs]

    # 🔥 remove duplicates safely
    seen = set()
    combined = []

    for doc in vec_docs + bm_docs:
        key = doc.strip().lower()
        if key not in seen:
            seen.add(key)
            combined.append(doc)

    return combined[:20]

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
from sentence_transformers import SentenceTransformer

# Re-initialize the model to ensure it's in scope
model = SentenceTransformer("all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import json
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

embedding_model = model

# Re-load error_db and populate error_lookup
with open("/content/drive/MyDrive/mosmb_rag/data/mosmb_error_structured.json") as f:
    error_db = json.load(f)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import json

with open("/content/drive/MyDrive/mosmb_rag/data/mosmb_error_structured.json") as f:
    error_db = json.load(f)

documents = []
metadatas = []
ids = []

for item in error_db:

    # combine into structured text
    doc_text = f"""
Error Code: {", ".join(item["error_codes"])}
Description: {item["error_string"]}
Solution:
{chr(10).join(item["solution"])}
"""

    for code in item["error_codes"]:
        documents.append(doc_text)

        metadatas.append({
            "type": "error",
            "error_codes": item["error_codes"],
            "category": item["category"]
        })

        ids.append(f"error_{code}")   # 🔥 CRITICAL FIX

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print("✅ Error data inserted correctly")

✅ Error data inserted correctly


In [ ]:
def get_error_from_chroma_fast(code):

    result = collection.get(ids=[f"error_{code}"])

    if not result["ids"]:
        return None

    doc = result["documents"][0]
    meta = result["metadatas"][0]

    return {
        "error_codes": meta["error_codes"],
        "error_string": doc.split("Description:")[1].split("Solution:")[0].strip(),
        "category": meta["category"],
        "solution": doc.split("Solution:")[1].strip().split("\n"),
    }

In [ ]:
def solve_error(query, codes):

    for code in codes:

        error = get_error_from_chroma_fast(code)

        if error:

            return {
                "type": "error",
                "data": error,
                "query": query
            }

    return None

In [ ]:
def solve_log(query):

    structured_query, codes, important_lines = process_log_input(query)

    # 🔥 PRIORITY 1 → error codes inside logs
    if codes:
        result = solve_error(query, codes)
        if result:
            return result

    # 🔥 fallback → treat as semantic issue
    return {
        "type": "log",
        "query": structured_query,
        "signals": important_lines
    }

In [ ]:
def rerank(query, docs):

    pairs = [[query, doc] for doc in docs]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, docs),
        reverse=True
    )

    # 🔥 STRONG FILTERING
    filtered = []

    for score, doc in ranked:
        if score > 0.3:   # threshold (tune later)
            filtered.append(doc)

    # fallback if too strict
    if len(filtered) < 3:
        filtered = [doc for _, doc in ranked[:5]]

    return filtered[:5]

In [ ]:
def format_error_output(error):

    print("\nERROR CODE:", ", ".join(error["error_codes"]))
    print("DESCRIPTION:", error["error_string"])
    print("CATEGORY:", error["category"])

    print("\nSOLUTION STEPS:")

    for i, step in enumerate(error["solution"], 1):
        print(f"{i}. {step}")

In [ ]:
def process_log_input(log_text):

    log_text_lower = log_text.lower()

    # ✅ USE ROBUST ERROR EXTRACTION (FIXED)
    codes = extract_error_codes(log_text)

    # 2. Extract severity-based lines (IMPORTANT)
    important_lines = []

    for line in log_text.split("\n"):
        if any(tag in line.upper() for tag in ["ERROR", "WARN"]):
            important_lines.append(line.strip())

    # 3. Extract keywords (signals)
    signals = []

    if "ldap" in log_text_lower:
        signals.append("LDAP failure")

    if "kerberos" in log_text_lower:
        signals.append("Kerberos issue")

    if "gssapi" in log_text_lower:
        signals.append("GSSAPI error")

    if "auth" in log_text_lower:
        signals.append("Authentication issue")

    # 4. Build structured query (ONLY for fallback use)
    structured_query = ""

    if signals:
        structured_query += "Issues detected: " + ", ".join(signals) + ". "

    if codes:
        structured_query += "Error codes: " + ", ".join(codes) + ". "

    structured_query += "Provide root cause and solution."

    # ⚠️ IMPORTANT CHANGE
    return log_text, codes, important_lines

In [ ]:
import re

def extract_error_codes(text):

    # Step 1: normalize
    text = text.upper()

    # Step 2: extract all possible patterns
    patterns = [
        r'\b[0-9A-F]{8}\b',          # 02010016
        r'\bEC[0-9A-F]{8}\b',       # EC02010016
        r'\[EC[0-9A-F]{8}\]',       # [EC02010016]
        r'\[[0-9A-F]{8}\]'          # [02010016]
    ]

    matches = []

    for pattern in patterns:
        found = re.findall(pattern, text)
        matches.extend(found)

    # Step 3: clean results → always return pure 8-digit code
    cleaned = []

    for m in matches:
        code = re.sub(r'[^0-9A-F]', '', m)  # remove EC, [, ]
        if len(code) >= 8:
            cleaned.append(code[-8:])  # last 8 digits

    return list(set(cleaned)) if cleaned else None

In [ ]:
extract_error_codes("""
[MO_ERROR]: EC02010016: Failed getpwnam
""")

['02010016']

In [ ]:
def detect_intent(query):

    import re

    q = query.lower()

    # 1. Detect error codes (handles EC02010016, 02010016, logs)
    codes = re.findall(r'(?:EC)?([0-9]{8})', query.upper())
    codes = list(set(codes))

    if codes:
        return {
            "type": "error",
            "codes": codes
        }

    # 2. Detect log pattern
    if "[" in query and "]" in query and ("error" in q or "warn" in q):
        return {
            "type": "log",
            "codes": None
        }

    # 3. Detect semantic error queries (no code but error intent)
    if "error" in q or "failed" in q or "issue" in q:
        return {
            "type": "error_semantic",
            "codes": None
        }

    # 4. Default → documentation query
    return {
        "type": "doc",
        "codes": None
    }

In [ ]:
print(detect_intent("02010016"))
print(detect_intent("EC02010016"))
print(detect_intent("[MO_ERROR]: something failed"))
print(detect_intent("I am facing kerberos error"))
print(detect_intent("How to configure kerberos?"))

('error', ['02010016'])
('error', ['02010016'])
('log', None)
('error_semantic', None)
('doc', None)


In [ ]:
def classify_query_type(query):

    q = query.lower()

    if "how" in q or "configure" in q or "install" in q:
        return "how_to"

    if "error" in q or "failed" in q or "not working" in q:
        return "troubleshoot"

    if "what" in q or "where" in q:
        return "info"

    if "step" in q or "explain" in q:
        return "followup"

    return "general"

In [ ]:
def build_doc_prompt(context, query):
    query_type = classify_query_type(query)

    return f"""
You are a MoSMB technical documentation assistant.

Your job:
Provide a clear, structured answer based ONLY on the given context.

STRICT RULES:
- Do NOT hallucinate
- Do NOT assume issues
- Do NOT add extra commentary
- Do NOT ask questions
- ONLY use relevant information
- IGNORE irrelevant lines from context

FORMAT RULE (MANDATORY):
- If the question is "how to" → output ONLY numbered steps
- Minimum 3 steps if applicable
- DO NOT add headings
- DO NOT number steps (system will format)
- Each step must be clear and actionable
- No paragraphs

QUERY TYPE: {query_type}

CONTEXT:
{context}

USER QUERY:
{query}

INSTRUCTIONS:

If QUERY TYPE = how_to:
→ Return ONLY actionable steps

If QUERY TYPE = troubleshoot:
→ Return:
Root Cause:
...

Solution:
- Step 1
- Step 2

If QUERY TYPE = info:
→ Return short direct answer (2–4 lines)

If QUERY TYPE = followup:
→ Explain ONLY the requested step clearly

OUTPUT:
"""

In [ ]:
def generate_answer(prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = llm.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.1,          # 🔥 reduce randomness
        top_p=0.9,
        do_sample=False,
        repetition_penalty=1.2,   # 🔥 stop looping
        eos_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

In [ ]:
def generate_error_answer(error_obj):

    error = error_obj["data"]
    query = error_obj["query"]

    steps = [s.strip() for s in error["solution"] if s.strip()]

    context = "\n".join([f"{i+1}. {step}" for i, step in enumerate(steps)])

    prompt = f"""
You are a MoSMB support system.

STRICT RULES (NON-NEGOTIABLE):
- You MUST use ONLY the given solution steps
- You MUST NOT add any new steps
- You MUST NOT remove any step
- You MUST NOT add placeholders like "refer documentation"
- You MUST NOT ask user anything
- You MUST NOT add explanations unless explicitly asked

User Query:
{query}

Error Code: {", ".join(error['error_codes'])}
Description: {error['error_string']}

Official Steps:
{context}

TASK:
Return the SAME steps cleanly.

OUTPUT FORMAT:
1. step
2. step
3. step
"""

    answer = generate_answer(prompt)

    return format_steps_output(answer)

In [ ]:
def generate_log_answer(log_obj):

    query = log_obj["query"]
    signals = log_obj["signals"]

    context = "\n".join(signals[:5])  # limit noise

    prompt = f"""
You are a MoSMB support engineer.

Analyze the log signals below and determine:

1. Root cause
2. Exact resolution steps

STRICT RULES:
- Use only relevant technical reasoning
- Do not hallucinate
- Keep answer precise
- No greetings or extra text

Log Signals:
{context}

User Query:
{query}

Output format:

Root Cause:
...

Solution:
- Step 1
- Step 2
"""

    answer = generate_answer(prompt)

    return clean_output(answer)

In [ ]:
results = collection.get(where={"type": "error"})
print(len(results["ids"]))

701


In [ ]:
test_error = solve_error(
    "I am facing EC02010016 error",
    ["02010016"]
)

print(generate_error_answer(test_error))

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Please provide more information on how to resolve the issue with the given error code (EC02010016). I would like to know what it means and how to correct it.
Response:
The error code EC02010016 indicates an issue while getting the PWNAM (Primary User Name) from the system. This can occur when there's a misconfiguration in the NSSWITCH configuration file or incorrect settings in the SSSD (System Security Services Daemon).
To resolve this issue, follow these official solution steps:
1. **For all Active Directory users**: Please ensure that your AD users have been properly synced with your Linux machine using tools such as `realmd` and `sssctl`.
2. **Ensure /etc/nsswitch.conf is configured correctly**: Verify that `/etc/nsswitch.conf` contains the following lines at the beginning:
```
passwd:     files compat authconfig_nis
shadow:    files compat authconfig_nis
group:      files compat authconfig_n


In [ ]:
log_test = solve_log("""[2026-03-23 17:52:39.031]:[MO_ERROR]:[SMBCORE]:[TID 177306]:[EC02010016]: Failed getpwnam. username = samikshad, errno is : 0 & Id mapper service not configured correctly""")
print(log_test)

{'type': 'error', 'data': {'error_codes': ['02010016'], 'error_string': 'Failed getpwnam. username = %s, errno is : %d & Id mapper service not configured correctly', 'category': 'COMMON', 'solution': ['For all Active Directory users 1', 'Ensure /etc/nsswitch.conf is configured correctly nsswitch.conf Configuration 2', 'Check sssd configurations w.r.t authentication mode required', 'krb5 - SSSD Configuration ntlmAD - SSSD Configuration For local linux users please check if nsswitch.conf passwd has files as first option.']}, 'query': '[2026-03-23 17:52:39.031]:[MO_ERROR]:[SMBCORE]:[TID 177306]:[EC02010016]: Failed getpwnam. username = samikshad, errno is : 0 & Id mapper service not configured correctly'}


In [ ]:
import re

def format_steps_output(text):

    lines = text.split("\n")

    cleaned = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # ❌ remove markdown headings / bold lines
        if "**" in line or "step-by-step" in line.lower():
            continue

        # ❌ remove existing numbering (1. , 2. , etc.)
        line = re.sub(r'^\d+\.\s*', '', line)

        # ❌ remove bullet points
        line = re.sub(r'^[-•]\s*', '', line)

        # keep only meaningful lines
        if len(line) > 20:
            cleaned.append(line)

    # ✅ rebuild clean numbered steps
    steps = []
    for i, line in enumerate(cleaned[:6], 1):
        steps.append(f"{i}. {line}")

    return "\n".join(steps)

In [ ]:
def rag_pipeline(query):

    # 🔥 STEP 1 — robust error detection (NEW)
    intent = detect_intent(query)

    # ✅ ERROR FLOW
    if intent["type"] in ["error", "error_semantic"]:

        result = solve_error(query, intent["codes"])

        if result:
            answer = generate_error_answer(result)
            print(answer)
            return
    # 🔥 STEP 2 — hybrid retrieval (UPDATED)
    docs = hybrid_retrieval(query)

    # 🔥 STEP 3 — rerank (UPDATED)
    ranked_docs = rerank(query, docs)

    # 🔥 STEP 4 — clean context (NEW)
    final_docs = clean_context(ranked_docs)

    # 🔥 STEP 5 — build context (FIXED)
    context = "\n\n".join(final_docs)

    # 🔥 STEP 6 — generate answer
    prompt = build_doc_prompt(context, query)

    answer = generate_answer(prompt)
    answer = clean_output(answer)

    # Only format steps if needed
    if "how" in query.lower():
        answer = format_steps_output(answer)
    print(answer)

In [ ]:
rag_pipeline("How to configure kerberos for mosmb cluster?")

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


1. Please provide instructions on how to configure Kerberos for MosMB Cluster.
2. Run `ldapwhoami` command on the MoSMB node to verify the Kerberos ticket.
3. Purge the Kerberos cache on the client-side using `klist purge`.
4. Configure the MoSMB server to use Kerberos authentication by setting the security option to 'kerberos' when mounting the share.
5. Verify that the FQDN of the MoSMB server is used while mounting the share.
6. Execute `/usr/bin/mosmb_passwd` binary to add Linux local users as MoSMB users.


In [ ]:
rag_pipeline("I am facing EC02010016 error, how to resolve?")

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


1. For all Active Directory users
2. Ensure /etc/nsswitch.conf is configured correctly nsswitch.conf Configuration
3. Check sssd configurations w.r.t authentication mode required
4. For local linux users please check if nsswitch.conf passwd has files as first option.


In [ ]:
rag_pipeline("Kerberos failure EC02010016 how to fix?")

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


1. For all Active Directory users
2. Ensure /etc/nsswitch.conf is configured correctly nsswitch.conf Configuration
3. Check sssd configurations w.r.t authentication mode required
4. krb5 - SSSD Configuration ntlmAD - SSSD Configuration For local linux users please check if nsswitch.conf passwd has files as first option.


In [ ]:
rag_pipeline("What ACLs are supported")

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Please provide an informative response regarding what ACLs are supported.
ANSWER:
NTACL rules are used to check access. In this case, Mo SMB uses the extended attributes provided by the file system for storing the ACLs. NTAcl is not supported on "Auth Mode": "ntlm Local". POSIX ACL authorization mode is also supported.


In [ ]:
rag_pipeline("Kerberos ticket generation failed")

TypeError: 'NoneType' object is not iterable

In [ ]:
rag_pipeline(
"what ACLs are suppored "
)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


1. Currently, MSDFS links created only on files are supported.
2. Directory-based links are not fully supported.
3. Verify if the msDfsSupport parameter is set to 'no' in mosmb.json; otherwise, the links might not be enumerable on the client-side.


In [ ]:
rag_pipeline(
"[2026-03-23 17:52:39.031]:[MO_ERROR]:[SMBCORE]:[TID 177306]:[EC02010016]: Failed getpwnam. username = samikshad, errno is : 0 & Id mapper service not configured correctly"
)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


1. For all Active Directory users
2. Ensure /etc/nsswitch.conf is configured correctly
3. Check sssd configurations w.r.t authentication mode required
4. For local linux users please check if nsswitch.conf passwd has files as first option.
